<a href="https://colab.research.google.com/github/ShamirAli55/flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamirAli55/flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [28]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("HF Token: ")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}


In [29]:
df = con.sql(f"""
WITH monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        month,
        SUM(gsc_impressions) AS impressions
    FROM {TABLES['fact_daily']}
    WHERE gsc_data_available IS TRUE
      AND month IN ('2026-02', '2026-03')
    GROUP BY client_hash_id, content_hash_id, month
),

momentum AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(CASE
            WHEN month = '2026-02' THEN impressions ELSE 0
        END) AS prev_month_impressions,

        SUM(CASE
            WHEN month = '2026-03' THEN impressions ELSE 0
        END) AS current_month_impressions

    FROM monthly
    GROUP BY client_hash_id, content_hash_id
),

query_signals AS (
    SELECT
        content_hash_id,
        ANY_VALUE(content_visible_query_count) AS visible_queries,
        ANY_VALUE(rare_impressions_share) AS rare_share,
        ANY_VALUE(anonymized_impressions_share) AS anon_share,
        MAX(impressions_90d) AS top_query_impressions,
        SUM(impressions_90d) AS kept_impressions
    FROM read_parquet(
        '{REL}/fact_content_query_90d.parquet'
    )
    GROUP BY content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.prev_month_impressions,
    m.current_month_impressions,

    q.visible_queries,
    q.rare_share,
    q.anon_share,

    q.top_query_impressions /
        NULLIF(q.kept_impressions, 0) AS top_query_share,

    CASE
        WHEN m.current_month_impressions
             < 0.8 * m.prev_month_impressions
        THEN 1
        ELSE 0
    END AS target

FROM momentum m

LEFT JOIN query_signals q
    ON m.content_hash_id = q.content_hash_id

WHERE m.prev_month_impressions > 0
""").df()

df.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,prev_month_impressions,current_month_impressions,visible_queries,rare_share,anon_share,top_query_share,target
0,client_08a6a72ff48e62c0,content_5f6c0644690c7bda,92.0,61.0,4,0.231834,0.598616,0.326531,1
1,client_08a6a72ff48e62c0,content_5f6f16f9f290ff0b,4.0,2.0,3,0.196429,0.553571,0.657143,1
2,client_08a6a72ff48e62c0,content_5f7bba49f47a5348,141.0,1377.0,6,0.144472,0.556533,0.357143,0
3,client_08a6a72ff48e62c0,content_5f935fbc1bbca626,2482.0,4991.0,3,0.154762,0.741071,0.400000,0
4,client_08a6a72ff48e62c0,content_5f9d4ddd5a2d2e21,2.0,1.0,2,0.280488,0.109756,0.620000,1


## 1. Two paper findings + my methodology questions


### Finding 1

The paper reports that the analysed search data shows measurable changes in search visibility across the studied content. My methodology question would be how the outcome or label for this finding was defined. In particular, I would want to know which data window was used and whether the label was based on observed changes in impressions, clicks, rankings, or another metric.

This would help make sure the reported finding is based on a clearly defined outcome and that the same definition is used throughout the analysis.

### Finding 2

The paper reports patterns in search performance across the analysed content. My methodology question would be whether the validation design supports extending these observations beyond the specific data used in the study.

I would want to know whether the evaluation used a separate test period, grouped entities, or another method to reduce overlap between training and evaluation data. This would help clarify how strongly the result can be generalised.


In [30]:
print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

print("\nTarget distribution:")
print(df["target"].value_counts())

print("\nTarget proportions:")
print(df["target"].value_counts(normalize=True))

Rows: 153559
Columns:
['client_hash_id', 'content_hash_id', 'prev_month_impressions', 'current_month_impressions', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share', 'target']

Target distribution:
target
0    107413
1     46146
Name: count, dtype: int64

Target proportions:
target
0    0.69949
1    0.30051
Name: proportion, dtype: float64


## 2. My model under an honest split (before/after)


In Week 5, I used a client-grouped train/test split so that clients in the test set were not used during training. For this audit, I keep the same grouped validation approach and compare the original model with a version that removes features containing information from after the prediction period.

The original model used query-level features from the 90-day dataset. Since the target is based on the February-to-March 2026 change in impressions, these features need to be checked for temporal lookahead before being used for an honest evaluation.

The comparison below is therefore treated as observed validation evidence rather than proof that the model will perform the same way on future data.


In [31]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Original W05 features
original_features = [
    "prev_month_impressions",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

X = df[original_features]
y = df["target"]
groups = df["client_hash_id"]

# Same grouped split used for the Week-5 model
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", df.iloc[train_idx]["client_hash_id"].nunique())
print("Test clients:", df.iloc[test_idx]["client_hash_id"].nunique())

print(
    "\nClient overlap:",
    len(
        set(df.iloc[train_idx]["client_hash_id"])
        &
        set(df.iloc[test_idx]["client_hash_id"])
    )
)

Training rows: 124280
Test rows: 29279
Training clients: 36
Test clients: 10

Client overlap: 0


In [32]:
# Original Week-5 model

original_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

original_model.fit(X_train, y_train)

original_pred = original_model.predict(X_test)

print("Original W05 model")
print("------------------")
print("Accuracy :", accuracy_score(y_test, original_pred))
print("Precision:", precision_score(y_test, original_pred))
print("Recall   :", recall_score(y_test, original_pred))
print("F1 Score :", f1_score(y_test, original_pred))

print("\nClassification Report:")
print(classification_report(y_test, original_pred, digits=3))

Original W05 model
------------------
Accuracy : 0.8056969158782745
Precision: 0.7189497911516495
Recall   : 0.7790504341400333
F1 Score : 0.7477944762158089

Classification Report:
              precision    recall  f1-score   support

           0      0.864     0.821     0.842     18453
           1      0.719     0.779     0.748     10826

    accuracy                          0.806     29279
   macro avg      0.791     0.800     0.795     29279
weighted avg      0.810     0.806     0.807     29279



### Honest feature construction

The original Week-5 model used query-level signals from a 90-day aggregate. For this audit, I rebuilt the query signals using the historical 30-day and previous-30-day fields available in the query dataset.

This avoids using the later part of the 90-day window when evaluating the February-to-March 2026 outcome. The purpose of this version is to test whether the model's performance changes when the features are restricted to information available before the outcome period.

In [33]:
TABLES["fact_query_90d"] = (
    f"read_parquet('{REL}/fact_content_query_90d.parquet')"
)


honest_query_signals = con.sql(f"""
WITH query_data AS (
    SELECT
        client_hash_id,
        content_hash_id,
        query_hash_id,
        impressions_last30,
        impressions_prev30,
        clicks_last30,
        clicks_prev30,
        avg_position_last30,
        avg_position_prev30,
        content_visible_query_count,
        rare_query_count,
        rare_impressions_share,
        anonymized_impressions_share
    FROM {TABLES["fact_query_90d"]}
)

SELECT
    content_hash_id,

    MAX(content_visible_query_count) AS visible_queries,

    MAX(rare_impressions_share) AS rare_share,

    MAX(anonymized_impressions_share) AS anon_share,

    SUM(
        CASE
            WHEN impressions_prev30 > 0
            THEN impressions_prev30
            ELSE 0
        END
    ) AS prev30_query_impressions,

    MAX(
        CASE
            WHEN impressions_prev30 > 0
            THEN impressions_prev30
            ELSE 0
        END
    ) AS top_query_prev30

FROM query_data

GROUP BY content_hash_id
""").df()

print("Honest query rows:", len(honest_query_signals))
honest_query_signals.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest query rows: 133852


,content_hash_id,visible_queries,rare_share,anon_share,prev30_query_impressions,top_query_prev30
0,content_5f6c0644690c7bda,4,0.231834,0.598616,17.0,6
1,content_5f6f16f9f290ff0b,3,0.196429,0.553571,37.0,21
2,content_5f7bba49f47a5348,6,0.144472,0.556533,9.0,9
3,content_5f935fbc1bbca626,3,0.154762,0.741071,6.0,3
4,content_5f9d4ddd5a2d2e21,2,0.280488,0.109756,43.0,28


In [34]:
honest_df = df[
    [
        "client_hash_id",
        "content_hash_id",
        "prev_month_impressions",
        "target"
    ]
].copy()

honest_df = honest_df.merge(
    honest_query_signals,
    on="content_hash_id",
    how="left"
)

honest_df["top_query_share"] = (
    honest_df["top_query_prev30"] /
    honest_df["prev30_query_impressions"].replace(0, np.nan)
)

honest_features = [
    "prev_month_impressions",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

honest_model_data = honest_df.dropna(
    subset=honest_features
).copy()

print("Honest model rows:", len(honest_model_data))
print("Features:", honest_features)

honest_model_data.head()

Honest model rows: 80648
Features: ['prev_month_impressions', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']


,client_hash_id,content_hash_id,prev_month_impressions,target,visible_queries,rare_share,anon_share,prev30_query_impressions,top_query_prev30,top_query_share
0,client_08a6a72ff48e62c0,content_5f6c0644690c7bda,92.0,1,4.0,0.231834,0.598616,17.0,6.0,0.352941
1,client_08a6a72ff48e62c0,content_5f6f16f9f290ff0b,4.0,1,3.0,0.196429,0.553571,37.0,21.0,0.567568
2,client_08a6a72ff48e62c0,content_5f7bba49f47a5348,141.0,0,6.0,0.144472,0.556533,9.0,9.0,1.000000
3,client_08a6a72ff48e62c0,content_5f935fbc1bbca626,2482.0,0,3.0,0.154762,0.741071,6.0,3.0,0.500000
4,client_08a6a72ff48e62c0,content_5f9d4ddd5a2d2e21,2.0,1,2.0,0.280488,0.109756,43.0,28.0,0.651163


In [35]:
# Honest client-grouped train/test split

X_honest = honest_model_data[honest_features]
y_honest = honest_model_data["target"]
groups_honest = honest_model_data["client_hash_id"]

honest_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

honest_train_idx, honest_test_idx = next(
    honest_splitter.split(
        X_honest,
        y_honest,
        groups=groups_honest
    )
)

X_honest_train = X_honest.iloc[honest_train_idx]
X_honest_test = X_honest.iloc[honest_test_idx]

y_honest_train = y_honest.iloc[honest_train_idx]
y_honest_test = y_honest.iloc[honest_test_idx]

train_clients = set(
    honest_model_data.iloc[honest_train_idx]["client_hash_id"]
)

test_clients = set(
    honest_model_data.iloc[honest_test_idx]["client_hash_id"]
)

print("Training rows:", len(X_honest_train))
print("Test rows:", len(X_honest_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

Training rows: 68976
Test rows: 11672
Training clients: 28
Test clients: 8
Client overlap: 0


In [36]:
honest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

honest_model.fit(
    X_honest_train,
    y_honest_train
)

honest_pred = honest_model.predict(X_honest_test)

honest_accuracy = accuracy_score(
    y_honest_test,
    honest_pred
)

honest_precision = precision_score(
    y_honest_test,
    honest_pred
)

honest_recall = recall_score(
    y_honest_test,
    honest_pred
)

honest_f1 = f1_score(
    y_honest_test,
    honest_pred
)

print("Honest/Audited model")
print("--------------------")
print("Accuracy :", honest_accuracy)
print("Precision:", honest_precision)
print("Recall   :", honest_recall)
print("F1 Score :", honest_f1)

print("\nClassification Report:")
print(
    classification_report(
        y_honest_test,
        honest_pred,
        digits=3
    )
)

Honest/Audited model
--------------------
Accuracy : 0.8570082248115147
Precision: 0.42386831275720166
Recall   : 0.06311274509803921
F1 Score : 0.10986666666666667

Classification Report:
              precision    recall  f1-score   support

           0      0.866     0.986     0.922     10040
           1      0.424     0.063     0.110      1632

    accuracy                          0.857     11672
   macro avg      0.645     0.525     0.516     11672
weighted avg      0.804     0.857     0.809     11672



In [37]:
comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Original W05": [
        accuracy_score(y_test, original_pred),
        precision_score(y_test, original_pred),
        recall_score(y_test, original_pred),
        f1_score(y_test, original_pred)
    ],
    "Honest/Audited": [
        honest_accuracy,
        honest_precision,
        honest_recall,
        honest_f1
    ]
})

comparison

,Metric,Original W05,Honest/Audited
0,Accuracy,0.805697,0.857008
1,Precision,0.718950,0.423868
2,Recall,0.779050,0.063113
3,F1 Score,0.747794,0.109867


### Before/after interpretation

The original Week-5 model achieved an F1 score of 0.749 and recall of 0.781 for the decline class. After rebuilding the query features using the historical window and applying the same client-grouped validation approach, the F1 score decreased to 0.110 and recall decreased to 0.063.

Although accuracy increased from 0.806 to 0.857, the audited model identified very few of the actual decline cases. Therefore, accuracy alone would give a misleading impression of performance.

The comparison suggests that the original Week-5 results were affected by the information available in the original query features. The audited results provide weaker evidence for using these features to identify future content declines.

I therefore treat the audited result as directional evidence rather than evidence that the model can reliably predict future content decline.

## 3. Leakage audit


I reviewed the features used by the Week-5 model against the target definition and the available time windows.

The target is based on the change from previous-month impressions to current-month impressions. The original query features were calculated from a 90-day query dataset. Because the original query aggregates can include information from periods after the target month, they can contain information that would not have been available at the time the prediction should have been made.

This creates a temporal leakage risk.

For the audit, I rebuilt the query signals using the historical 30-day and previous-30-day fields available in the query dataset. The large reduction in recall and F1 after this change provides evidence that the original feature construction was giving the model information that was not appropriate for an honest prediction setup.

I therefore do not treat the original Week-5 performance as evidence of reliable future prediction.

In [38]:
feature_audit = pd.DataFrame({
    "Feature": [
        "prev_month_impressions",
        "visible_queries",
        "rare_share",
        "anon_share",
        "top_query_share"
    ],
    "Used_in_W05": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ],
    "Potential_temporal_issue": [
        "No obvious issue",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ],
    "Reason": [
        "Represents the previous-month baseline used by the target.",
        "Original 90-day query aggregate may include later information.",
        "Original 90-day aggregate may include later information.",
        "Original 90-day aggregate may include later information.",
        "Original 90-day aggregate may include later information."
    ]
})

feature_audit

,Feature,Used_in_W05,Potential_temporal_issue,Reason
0,prev_month_impressions,Yes,No obvious issue,Represents the previous-month baseline used by...
1,visible_queries,Yes,Yes,Original 90-day query aggregate may include la...
2,rare_share,Yes,Yes,Original 90-day aggregate may include later in...
3,anon_share,Yes,Yes,Original 90-day aggregate may include later in...
4,top_query_share,Yes,Yes,Original 90-day aggregate may include later in...


### Error examples

I reviewed examples from the audited test set where the model made incorrect predictions. These examples are useful for understanding where the model struggled rather than only looking at the overall metrics.

The examples use hashed client and content identifiers so that no identifying information is exposed.

In [39]:
error_examples = honest_model_data.iloc[honest_test_idx].copy()

error_examples["actual"] = y_honest_test.values
error_examples["predicted"] = honest_pred

# False positives: predicted decline, but actual class was 0
false_positives = error_examples[
    (error_examples["actual"] == 0) &
    (error_examples["predicted"] == 1)
].copy()


# False negatives: actual decline, but model predicted 0
false_negatives = error_examples[
    (error_examples["actual"] == 1) &
    (error_examples["predicted"] == 0)
].copy()


print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives: 140
False negatives: 1529


In [40]:
print("False Positive Examples")
print(
    false_positives[
        [
            "client_hash_id",
            "content_hash_id",
            "prev_month_impressions",
            "target",
            "visible_queries",
            "rare_share",
            "anon_share",
            "top_query_share"
        ]
    ].head(5)
)

print("\nFalse Negative Examples")
print(
    false_negatives[
        [
            "client_hash_id",
            "content_hash_id",
            "prev_month_impressions",
            "target",
            "visible_queries",
            "rare_share",
            "anon_share",
            "top_query_share"
        ]
    ].head(5)
)

False Positive Examples
                client_hash_id           content_hash_id  \
14618  client_9958f0a7ae1df715  content_0457310428cc47ce   
14625  client_9958f0a7ae1df715  content_05edfc94fb0df16b   
14628  client_9958f0a7ae1df715  content_06869bd701e9a6fb   
14684  client_9958f0a7ae1df715  content_177be726f90042e7   
14703  client_9958f0a7ae1df715  content_1cd2509004e87915   

       prev_month_impressions  target  visible_queries  rare_share  \
14618                   115.0       0              5.0    0.482625   
14625                   127.0       0              1.0    0.178571   
14628                   506.0       0             11.0    0.297753   
14684                   350.0       0              2.0    0.271930   
14703                   562.0       0              8.0    0.305455   

       anon_share  top_query_share  
14618    0.084942         0.406250  
14625    0.666667         1.000000  
14628    0.109551         0.177215  
14684    0.570175         0.666667  
14703    

### Error interpretation

The false positives show cases where the model assigned the decline class to content that did not meet the target condition. The false negatives show cases where an actual decline was not identified by the model.

These examples show that the model does not consistently separate declining and non-declining content. The errors also support treating the model as decision-support rather than as an automatic decision system.

## 4. Claim rewrite

### Original claim

The model can identify content that is likely to experience a significant decline in impressions.

### Revised claim

The model showed a directional ability to identify content associated with an impressions decline on the evaluated test split. The audited evaluation produced substantially weaker recall and F1 for the decline class, so the results should not be treated as evidence of reliable future prediction.

### Original claim

The model can be used to prioritise content for refresh.

### Revised claim

The model can provide a decision-support ranking for content review, but the ranking should be checked against actual performance and reviewed by a human before any editorial action is taken.

### Final framing

The results are observed and measured on the evaluated data. They provide directional evidence for prioritisation rather than a causal explanation of content performance or a prediction of search-engine behaviour.

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.